In [5]:
import re
from collections.abc import Iterable

import httpx
import pandas as pd

In [6]:
TENCENT_HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Referer": "https://stockapp.finance.qq.com/mstats/",
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Safari/537.36"
    ),
}

client = httpx.Client(
    timeout=httpx.Timeout(connect=5.0, read=15.0, write=15.0, pool=15.0),
    limits=httpx.Limits(max_keepalive_connections=0, max_connections=1),
    trust_env=False,
    follow_redirects=True,
    headers=TENCENT_HEADERS,
)


### 腾讯行情中心

对应地址 - [腾讯行情中心](https://stockapp.finance.qq.com/mstats/)

> **接口边界（2026-08-26 核验）**
> 不能把网页菜单名称直接当成 `getBoardRankList` 的 `board_code`。当前网页的沪深京、创业板、科创板股票列表才调用 `https://proxy.finance.qq.com/cgi/cgi-bin/rank/hs/getBoardRankList`；把 `hy` 或 `hy2` 传给这条接口会返回 `code=0` 但 `data.total=0`，所以它们不是这条接口的申万参数。

#### 沪深京、创业板、科创板股票列表
对应地址 - `https://proxy.finance.qq.com/cgi/cgi-bin/rank/hs/getBoardRankList`

如果是申万行业分类不能用这个接口，会返回空列表结果。

`_appver`是网页客户端版本，浏览器目前取值`11.17.0`，`board_code`控制返回的股票列表所属板块，`aStock`表示沪深京A股、`cyb`表示创业板、`ksh`表示科创板。`sort_type` 点明排序字段，可以取`price` 最新价；`priceRatio` 涨跌幅；`priceChange` 涨跌额；`exchange` 换手率；`netMainIn` 主力净流入；`volumeRatio` 量比；`amplitude` 振幅；`volume` 成交量；`turnover` 成交额。配合排序方向`direct`， `down` 降序和`up` 升序。 `offset`和`count`分别表示分页起始偏移量和单页记录数。

#### 申万行业分类

腾讯行情中心的申万一级/二级**行业排行**使用另一条 GET 接口：`https://proxy.finance.qq.com/cgi/cgi-bin/rank/pt/getRank`。这里的参数名是 `board_type`：`hy` 表示申万一级行业，`hy2` 表示申万二级行业；`sort_type` 可用 `price`、`priceRatio`、`priceRatioD5`、`priceRatioD20`、`priceRatioD60`、`priceRatioW52`、`priceRatioY`，`direct`、`offset`、`count` 的含义与上面相同。
其中 `priceRatio` 是当日涨跌幅；`priceRatioD5`、`priceRatioD20`、`priceRatioD60` 分别是近 5、20、60 个交易日的累计涨跌幅；`priceRatioW52` 是近 52 周涨跌幅；`priceRatioY` 是年初至今涨跌幅（YTD）。这些字段的结果单位都是百分比，正数表示上涨，负数表示下跌。

这条 `rank/pt/getRank` 返回的是行业聚合行（例如 `stock_type=BK-HY-1/2`），不是行业成分股列表。因此要区分两件事：如果只需要复现腾讯页面的申万行业排行，调用 `rank/pt/getRank`；如果要给个股打申万行业标签或构建按行业筛选的股票池，应使用申万宏源的分类文件 `https://www.swsresearch.com/swindex/pdf/SwClass2021/StockClassifyUse_stock.xls`，不能用 `rank/hs/getBoardRankList` 代替。

#### ETF 标的池：腾讯当前接口不提供

当前 `getBoardRankList` 没有网页使用的 ETF `board_code`；把 `etf`、`fund`、`fund_etf` 等名称传入会得到空列表。腾讯旧版页面代码中出现过 `stockqt.gtimg.cn/...id=503`、`stock.gtimg.cn/data/index.php?...ranketf` 等基金列表地址，但当前不可作为可用的 ETF 分页接口，本文不再使用。


In [7]:
MARKET_LIST_ENDPOINT = "https://proxy.finance.qq.com/cgi/cgi-bin/rank/hs/getBoardRankList"

MARKET_PAGE_PARAMS = {
    "_appver": "11.17.0",
    "board_code": "aStock",
    "sort_type": "priceRatio",
    "direct": "down",
    "offset": 0,
    "count": 200,
}

market_page_resp = client.get(MARKET_LIST_ENDPOINT, params=MARKET_PAGE_PARAMS)
market_page_resp.raise_for_status()

market_page_json = market_page_resp.json()
market_page_json

{'code': 0,
 'msg': 'ok',
 'data': {'rank_list': [{'code': 'sh688835',
    'hsl': '49.05',
    'lb': '0.00',
    'ltsz': '56.25',
    'name': 'C高凯',
    'pe_ttm': '162.28',
    'pn': '13.41',
    'speed': '0.69',
    'state': '',
    'stock_type': 'GP-A-KCB',
    'turnover': '252915',
    'volume': '92132.82',
    'zd': '64.48',
    'zdf': '27.44',
    'zdf_d10': '388.07',
    'zdf_d20': '388.07',
    'zdf_d5': '388.07',
    'zdf_d60': '388.07',
    'zdf_w52': '388.07',
    'zdf_y': '388.07',
    'zf': '25.45',
    'zljlr': '18715.90',
    'zllc': '152608.91',
    'zllc_d5': '189547.98',
    'zllr': '171324.81',
    'zllr_d5': '429348.94',
    'zsz': '299.33',
    'zxj': '299.48'},
   {'code': 'sz301177',
    'hsl': '1.90',
    'lb': '2.22',
    'ltsz': '114.92',
    'name': '迪阿股份',
    'pe_ttm': '73.73',
    'pn': '1.89',
    'speed': '0.00',
    'state': '',
    'stock_type': 'GP-A-CYB',
    'turnover': '20349',
    'volume': '76027.00',
    'zd': '4.79',
    'zdf': '20.01',
    'zdf

In [12]:
board_code_probe = []
for probe_board_code in ["hy", "hy2", "etf"]:
    probe_params = {
        **MARKET_PAGE_PARAMS,
        "board_code": probe_board_code,
        "offset": 0,
        "count": 1,
    }
    probe_resp = client.get(MARKET_LIST_ENDPOINT, params=probe_params)
    probe_resp.raise_for_status()
    probe_json = probe_resp.json()
    probe_data = probe_json.get("data") or {}
    board_code_probe.append(
        {
            "board_code": probe_board_code,
            "code": probe_json.get("code"),
            "total": probe_data.get("total"),
            "rows": len(probe_data.get("rank_list") or []),
        }
    )

pd.DataFrame(board_code_probe)

,board_code,code,total,rows
0,hy,0,0,0
1,hy2,0,0,0
2,etf,0,0,0


In [14]:
INDUSTRY_RANK_ENDPOINT = "https://proxy.finance.qq.com/cgi/cgi-bin/rank/pt/getRank"

INDUSTRY_PAGE_PARAMS = {
    "board_type": "hy",
    "sort_type": "priceRatio",
    "direct": "down",
    "offset": 0,
    "count": 20,
}

industry_page_resp = client.get(INDUSTRY_RANK_ENDPOINT, params=INDUSTRY_PAGE_PARAMS)
industry_page_resp.raise_for_status()
industry_page_json = industry_page_resp.json()
industry_page_json

{'code': 0,
 'msg': 'ok',
 'data': {'rank_list': [{'code': 'pt01801790',
    'hsl': '1.23',
    'lb': '1.94',
    'ltsz': '55126.51',
    'lzg': {'code': 'sz000712',
     'name': '锦龙股份',
     'zd': '0.89',
     'zdf': '10.05',
     'zxj': '9.75'},
    'name': '非银金融',
    'speed': '-0.01',
    'stock_type': 'BK-HY-1',
    'turnover': '5950437',
    'volume': '51652049.00',
    'zd': '42.94',
    'zdf': '2.47',
    'zdf_d20': '-0.17',
    'zdf_d5': '4.25',
    'zdf_d60': '3.97',
    'zdf_w52': '-16.39',
    'zdf_y': '-13.22',
    'zgb': '77/79',
    'zljlr': '575651.02',
    'zljlr_d20': '-1873781.84',
    'zljlr_d5': '667355.10',
    'zllc': '2294262.52',
    'zllr': '2869913.54',
    'zsz': '70072.32',
    'zxj': '1782.56'},
   {'code': 'pt01801050',
    'hsl': '3.28',
    'lb': '1.15',
    'ltsz': '46418.44',
    'lzg': {'code': 'sz002295',
     'name': '精艺股份',
     'zd': '0.92',
     'zdf': '10.03',
     'zxj': '10.09'},
    'name': '有色金属',
    'speed': '0.00',
    'stock_type': 'BK-

#### 单个标的行情快照

对应地址 - `https://sqt.gtimg.cn/utf8/?q=<symbol>&fmt=json`

[腾讯个股详情页](https://gu.qq.com/sh600519/gp) 当前加载的脚本把行情主机设为 `https://sqt.gtimg.cn`，并按 `/<encode>/?q=<symbols>&fmt=json` 构造请求；个股页面传入 `encode="utf8"`，所以实际地址就是 `https://sqt.gtimg.cn/utf8/?q=<symbol>&fmt=json`。
这和腾讯行情中心列表页使用的 `https://qt.gtimg.cn/q=<symbol>` 是两个网页入口：`qt` 返回 GBK 编码的 `v_<symbol>="字段~字段...";` 文本，`sqt/utf8/?fmt=json` 返回 UTF-8 JSON 对象。两者返回的行情数组字段基本一致。

`<symbol>` 需要交易所前缀和 6 位代码，例如 `sh515080`；多个标的用逗号拼接，例如 `sh515080,sh600519`。ETF 也可以按已知代码查询，但这条接口只查快照，不提供 ETF 标的池。`fmt=json` 控制返回 JSON；`utf8` 控制编码。这里不需要 `r`，`r` 主要是旧 `qt` 页面请求中用 `Math.random()` 生成的缓存绕过参数。

In [15]:
SINGLE_QUOTE_ENDPOINT = "https://sqt.gtimg.cn/utf8/"
tencent_symbol = "sh515080"
tencent_quote_resp = client.get(
    SINGLE_QUOTE_ENDPOINT,
    params={"q": tencent_symbol, "fmt": "json"},
)
tencent_quote_resp.raise_for_status()

<Response [200 OK]>

In [18]:
print(tencent_quote_resp.headers.get("content-type"))
tencent_quote_json = tencent_quote_resp.json()
tencent_quote_json

text/html; charset=utf8


{'sh515080': ['1',
  '中证红利ETF招商',
  '515080',
  '1.606',
  '1.594',
  '1.590',
  '2192007',
  '1313105',
  '878902',
  '1.605',
  '6444',
  '1.604',
  '10280',
  '1.603',
  '8205',
  '1.602',
  '9059',
  '1.601',
  '13610',
  '1.606',
  '20429',
  '1.607',
  '17123',
  '1.608',
  '63907',
  '1.609',
  '19689',
  '1.610',
  '52628',
  '',
  '20260826161433',
  '0.012',
  '0.75',
  '1.607',
  '1.585',
  '1.606/2192007/351172201',
  '2192007',
  '35117',
  '3.08',
  '',
  '',
  '1.607',
  '1.585',
  '1.38',
  '114.19',
  '114.19',
  '0.00',
  '1.753',
  '1.435',
  '0.89',
  '-126178',
  '1.602',
  '',
  '',
  '',
  '',
  '',
  '35117.2201',
  '23.0943',
  '1438',
  '   A',
  'ETF',
  '6.01',
  '2.03',
  '',
  '',
  '',
  '1.661',
  '1.401',
  '2.62',
  '2.88',
  '1.39',
  '7110148900',
  '7110148900',
  '-57.00',
  '5.38',
  '7110148900',
  '0.12',
  '1.6041',
  '4.42',
  '0.00',
  '1.5932',
  'CNY',
  '0',
  '___D__F__N',
  '1.615',
  '-24580',
  '']}

#### 返回字段顺序

`tencent_quote_fields` 是按位置返回的行情数组，当前样例共有 88 个元素（下标 0–87），接口本身不返回字段名。下面按数组下标说明；股票、ETF、指数等证券类型在部分位置的口径可能不同，空字符串表示腾讯没有提供该项，标记为“未公开/保留”的位置不要直接作为稳定业务字段使用。

`0` 是市场/行情类型代码，`1` 是证券名称，`2` 是证券代码，`3` 是最新价，`4` 是昨收价，`5` 是今开价，`6` 是成交量（股票和 ETF 通常按手，指数等类型单位可能不同），`7` 是外盘成交量/主动买入量，`8` 是内盘成交量/主动卖出量，`9` 和 `10` 分别是买一价、买一量，`11` 和 `12` 分别是买二价、买二量，`13` 和 `14` 分别是买三价、买三量，`15` 和 `16` 分别是买四价、买四量，`17` 和 `18` 分别是买五价、买五量，`19` 和 `20` 分别是卖一价、卖一量，`21` 和 `22` 分别是卖二价、卖二量，`23` 和 `24` 分别是卖三价、卖三量，`25` 和 `26` 分别是卖四价、卖四量，`27` 和 `28` 分别是卖五价、卖五量，`29` 是行情更新标志（`Uptodate`，部分返回为空），`30` 是行情更新时间，`31` 是涨跌额，`32` 是涨跌幅，`33` 是当日最高价，`34` 是当日最低价。

`35` 是由最新价、成交量和成交额组成的组合字段，格式通常为 `price/volume/amount`，`36` 是成交量的标准化/重复字段，`37` 是成交额（通常按万元表示），`38` 是换手率，`39` 是市盈率（通常对应 TTM），`40` 是证券状态码（空值通常表示正常），`41` 和 `42` 是备用/重复的最高价、最低价字段，当前 HS 样本通常分别与 `33`、`34` 相同，腾讯前端没有单独公开其稳定业务名称，`43` 是振幅，`44` 是流通市值（通常按亿元表示），`45` 是总市值（通常按亿元表示），`46` 是市净率，`47` 是涨停价，`48` 是跌停价，`49` 是量比，`50` 是五档委差（五档买量合计减去五档卖量合计），`51` 是均价，`52` 是动态市盈率，`53` 是上一年度/历史市盈率，`54` 和 `55` 是当前 HS 返回中通常为空的保留字段，`56` 是 Beta 系数，`57` 是精确成交额（通常按万元表示），`58` 是盘后成交额，`59` 是盘后成交量。

`60` 是证券属性/类别附加标识（当前样例为 `A`，腾讯未公开统一字段名），`61` 是证券类型（例如 `GP-A`、`ETF`、`ZS`），`62` 是年初至今涨跌幅，`63` 是近 5 个交易日涨跌幅，`64` 是股息率，`65` 和 `66` 是腾讯 HS 返回的其他区间指标，当前详情页前端未公开其稳定字段名和具体区间，`67` 和 `68` 分别是 52 周最高价、52 周最低价，`69`、`70`、`71` 分别是近 10、20、60 个交易日涨跌幅，`72` 是流通股本/流通股份数，`73` 是总股本/总股份数，`74` 是委比，`75` 是额外区间涨跌幅字段，当前 HS 前端未公开其具体区间口径，`76` 是调整后的流通股本字段，具体口径未公开，`77` 是 ETF 溢价率/折价率，`78` 是 ETF IOPV/参考净值，`79` 是 52 周涨跌幅，`80` 是涨速，`81` 是基金净值（ETF 等基金类型使用），`82` 是货币类型（例如 `CNY`），`83` 是做市商/市场做市标志，`84` 是证券属性标志串，`85`、`86`、`87` 是当前 HS 前端未公开或未使用的尾部字段。

其中 `6` 与 `36` 都是成交量相关字段，`37` 与 `57` 都是成交额相关字段但精度/口径不同；`69`–`71` 可与榜单接口的 `zdf_d10`、`zdf_d20`、`zdf_d60` 对应。这个数组是腾讯网页内部的位置协议，不同证券类型可能出现空值、重复值或字段口径变化。

#### 完整 88 位 mapping

`tencent_quote_fields` 是按位置返回的行情数组。以下不合并任何下标，`0` 到 `87` 每个位置各对应一个字段；字段名是根据腾讯详情页前端映射、公开行情库和样本交叉核对整理的，`Unknown`/`Reserved` 表示腾讯当前没有公开稳定含义。

```text
0: Mkt — 市场/行情类型代码
1: Name — 证券名称
2: Symbol — 证券代码
3: Price — 最新价
4: PrevClose — 昨收价
5: Open — 今开价
6: Vol — 成交量（股票和 ETF 通常按手，指数等类型单位可能不同）
7: OB — 外盘成交量/主动买入量
8: IB — 内盘成交量/主动卖出量
9: BP1 — 买一价
10: BS1 — 买一量
11: BP2 — 买二价
12: BS2 — 买二量
13: BP3 — 买三价
14: BS3 — 买三量
15: BP4 — 买四价
16: BS4 — 买四量
17: BP5 — 买五价
18: BS5 — 买五量
19: AP1 — 卖一价
20: AS1 — 卖一量
21: AP2 — 卖二价
22: AS2 — 卖二量
23: AP3 — 卖三价
24: AS3 — 卖三量
25: AP4 — 卖四价
26: AS4 — 卖四量
27: AP5 — 卖五价
28: AS5 — 卖五量
29: Uptodate — 行情更新/最近逐笔成交标志，当前样例常为空
30: TimeStamp — 行情更新时间，格式通常为 YYYYMMDDHHMMSS
31: Chg — 涨跌额
32: ChgRatio — 涨跌幅（%）
33: High — 当日最高价
34: Low — 当日最低价
35: Nominal/_Price/mins — 最新价/成交量/成交额组合字段，格式通常为 price/volume/amount
36: _Vol — 成交量的标准化/重复字段
37: Turnover — 成交额，通常按万元表示，精度较低
38: TurnoverRate — 换手率（%）
39: PE — 市盈率，通常对应 TTM
40: Status — 证券状态码，空值通常表示正常
41: _High — 备用/重复最高价字段，当前 HS 样本通常与 33 相同
42: _Low — 备用/重复最低价字段，当前 HS 样本通常与 34 相同
43: Amp — 振幅（%）
44: CSIC/ltz — 流通市值，通常按亿元表示
45: CS/zsz — 总市值，通常按亿元表示
46: _Change/sjl — 市净率（PB）
47: LimitUp — 涨停价
48: LimitDown — 跌停价
49: VolRate — 量比
50: OrderImbalance — 五档委差，即五档买量合计减去五档卖量合计
51: AvgPrice — 均价
52: DynamicRatio — 动态市盈率
53: LYRRatio — 上一年度/历史市盈率
54: Reserved54 — 当前 HS 返回中通常为空的保留字段，具体含义未公开
55: Reserved55 — 当前 HS 返回中通常为空的保留字段，具体含义未公开
56: Beta — Beta 系数
57: CJE — 精确成交额，通常按万元表示
58: PHCJE — 盘后成交额
59: PHCJL — 盘后成交量
60: AttributeExtra — 证券属性/类别附加标识，当前样例常见为 A，统一字段名未公开
61: StockType — 证券类型，例如 GP-A、ETF、ZS
62: NCZJ/ZDF_Y — 年初至今涨跌幅（%）
63: ZDF_D5 — 近 5 个交易日涨跌幅（%）
64: GXL — 股息率（%）
65: Unknown65 — 其他区间指标，当前详情页前端未公开稳定字段名和具体区间
66: Unknown66 — 其他区间指标，当前详情页前端未公开稳定字段名和具体区间
67: Week52High — 52 周最高价
68: Week52Low — 52 周最低价
69: ZDF_D10 — 近 10 个交易日涨跌幅（%）
70: ZDF_D20 — 近 20 个交易日涨跌幅（%）
71: ZDF_D60 — 近 60 个交易日涨跌幅（%）
72: LTGB — 流通股本/流通股份数
73: ZGB — 总股本/总股份数
74: WBCALE — 委比（%）
75: Unknown75 — 额外区间涨跌幅字段，当前 HS 前端未公开具体区间口径
76: LTGB_TZ — 调整后的流通股本字段，具体口径未公开
77: YZL — ETF 溢价率/折价率（%）
78: IOPV — ETF IOPV/参考净值
79: ZDF_W52 — 52 周涨跌幅（%）
80: Speed — 涨速
81: JZ — 基金净值，ETF 等基金类型使用
82: HBLX — 货币类型，例如 CNY
83: MarketMaker — 做市商/市场做市标志
84: Attribute — 证券属性标志串，例如 ___D__F__N
85: Unknown85 — 当前 HS 前端未公开的尾部字段
86: Unknown86 — 当前 HS 前端未公开的尾部字段
87: Unknown87 — 当前 HS 前端未公开的尾部字段
```

`6` 与 `36` 都是成交量相关字段，`37` 与 `57` 都是成交额相关字段但精度/口径不同；`69`–`71` 可与榜单接口的 `zdf_d10`、`zdf_d20`、`zdf_d60` 对应。字段名中带有 `Unknown`、`Reserved` 或“具体口径未公开”的位置，应保留原始值，不要在解析函数中强行转换成确定业务字段。